# BEIR — Đánh giá zero-shot các mô hình truy xuất thông tin

Chạy trên **Google Colab**: Runtime → Change runtime type → **T4 GPU**.

Pipeline: tải dataset BEIR → **BM25** → **dense retrieval** → **cross-encoder re-ranking** → đo **nDCG@10**, Recall@100 → tổng hợp bảng.

> BEIR là benchmark zero-shot → **không train**, chỉ đánh giá mô hình pretrained trên domain lạ.

In [ ]:
# Neu mang co proxy/chung chi tu ky (SSLError khi tai dataset/model tu HuggingFace) —
# block duoi vao ho tro truong hop do. Tren Google Colab thi vo hai (khong anh huong).
import ssl as _ssl

def _trust_all_ssl():
    _orig = _ssl.create_default_context

    def _patched(purpose=_ssl.Purpose.SERVER_AUTH, *a, **kw):
        ctx = _orig(purpose, *a, **kw)
        ctx.check_hostname = False
        ctx.verify_mode = _ssl.CERT_NONE
        return ctx

    _ssl.create_default_context = _patched

try:
    import requests
    requests.head('https://huggingface.co', timeout=5)
    print('Mang: OK (khong can bypass SSL)')
except Exception:
    try:
        requests.head('https://huggingface.co', timeout=5, verify=False)
        _trust_all_ssl()
        print('Mang: SSLError do cert tu ky -> da bat che do bypass SSL')
        import urllib3
        urllib3.disable_warnings()
    except Exception:
        print('Mang: khong truy cap duoc huggingface.co — kiem tra ket noi!')
        raise

import torch

In [ ]:
# Cell 2 — chuẩn bị thư mục
import os
DATA_DIR = '/content/data'
RESULTS_DIR = '/content/results'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print('OK')

In [ ]:
# Cell 3 — các hàm dùng chung (bản rút gọn của utils.py)
import gzip, json, math, re, zipfile
import requests

DATASET_URL = 'https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{name}.zip'

def _fetch(url):
    # Neu mang/proxy dung cert tu ky (SSLError) thi thu lai bo kiem tra SSL
    try:
        r = requests.get(url, stream=True, timeout=300)
        r.raise_for_status()
        return r
    except requests.exceptions.SSLError:
        import urllib3
        urllib3.disable_warnings()
        print('  [!] SSL chan -> thu lai bo kiem tra chung chi')
        r = requests.get(url, stream=True, timeout=300, verify=False)
        r.raise_for_status()
        return r

def download_dataset(name):
    out_dir = os.path.join(DATA_DIR, name)
    if os.path.exists(os.path.join(out_dir, 'corpus.jsonl')):
        return out_dir
    zip_path = os.path.join(DATA_DIR, f'{name}.zip')
    url = DATASET_URL.format(name=name)
    print('[download]', url)
    r = _fetch(url)
    with open(zip_path, 'wb') as f:
        for chunk in r.iter_content(chunk_size=1 << 20):
            f.write(chunk)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(DATA_DIR)
    os.remove(zip_path)
    return out_dir

def _open_maybe_gz(path):
    if path.endswith('.gz'):
        return gzip.open(path, 'rt', encoding='utf-8')
    return open(path, 'r', encoding='utf-8')

def _find_file(folder, stem):
    for suffix in ('', '.gz'):
        p = os.path.join(folder, stem + suffix)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(stem + ' trong ' + folder)

def load_corpus(folder):
    corpus = {}
    with _open_maybe_gz(_find_file(folder, 'corpus.jsonl')) as f:
        for line in f:
            if line.strip():
                row = json.loads(line)
                corpus[row['_id']] = (row.get('title', ''), row.get('text', ''))
    return corpus

def load_queries(folder):
    queries = {}
    with _open_maybe_gz(_find_file(folder, 'queries.jsonl')) as f:
        for line in f:
            if line.strip():
                row = json.loads(line)
                queries[row['_id']] = row.get('text', '')
    return queries

def load_qrels(folder, split='test'):
    qrels = {}
    with open(os.path.join(folder, 'qrels', split + '.tsv'), encoding='utf-8') as f:
        next(f)
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) == 3:
                qrels.setdefault(parts[0], {})[parts[1]] = int(float(parts[2]))
    return qrels

def filter_queries_by_qrels(queries, qrels):
    return {q: t for q, t in queries.items() if q in qrels}

def dcg_at_k(rels, k):
    return sum((2 ** r - 1) / math.log2(i + 2) for i, r in enumerate(rels[:k]))

def ndcg_at_k(results, qrels, k=10):
    scores = []
    for qid, ranked in results.items():
        gold = qrels.get(qid, {})
        if not gold:
            continue
        gained = [gold.get(d, 0) for d in ranked[:k]]
        ideal = sorted(gold.values(), reverse=True)[:k]
        idcg = dcg_at_k(ideal, k)
        scores.append(dcg_at_k(gained, k) / idcg if idcg > 0 else 0.0)
    return sum(scores) / len(scores) if scores else 0.0

def recall_at_k(results, qrels, k=100):
    scores = []
    for qid, ranked in results.items():
        gold = qrels.get(qid, {})
        if not gold:
            continue
        hits = sum(1 for d in ranked[:k] if d in gold)
        scores.append(hits / len(gold))
    return sum(scores) / len(scores) if scores else 0.0

def evaluate(results, qrels, name):
    nd = round(ndcg_at_k(results, qrels, 10), 4)
    rc = round(recall_at_k(results, qrels, 100), 4)
    print(f'[{name}] nDCG@10 = {nd:.4f} | Recall@100 = {rc:.4f}')
    return {'nDCG@10': nd, 'Recall@100': rc}

def save_result(dataset, model_name, metrics):
    path = os.path.join(RESULTS_DIR, f'{dataset}__{model_name}.json')
    with open(path, 'w') as f:
        json.dump({'dataset': dataset, 'model': model_name, **metrics}, f, indent=2)
    print('[save]', path)

def tokenize(text):
    return re.findall(r'[a-z0-9]+', text.lower())

print('utils san sang')

In [ ]:
# Cell 4 — chọn dataset và tải (thêm 'fiqa' khi đã chạy quen)
DATASETS = ['scifact', 'nfcorpus']
folders = {}
for ds in DATASETS:
    folders[ds] = download_dataset(ds)
    print(ds, '->', folders[ds])

In [ ]:
# Cell 5 — BM25 (CPU, vài phút)
from rank_bm25 import BM25Okapi

def run_bm25(ds):
    folder = folders[ds]
    corpus = load_corpus(folder)
    qrels = load_qrels(folder)
    queries = filter_queries_by_qrels(load_queries(folder), qrels)
    doc_ids = list(corpus)
    bm25 = BM25Okapi([tokenize(f'{t} {x}') for t, x in (corpus[d] for d in doc_ids)])
    results = {}
    for n, (qid, qtext) in enumerate(queries.items(), 1):
        scores = bm25.get_scores(tokenize(qtext))
        top = sorted(range(len(scores)), key=lambda j: scores[j], reverse=True)[:1000]
        results[qid] = [doc_ids[j] for j in top]
        if n % 100 == 0:
            print(f'  {n}/{len(queries)}')
    m = evaluate(results, qrels, f'BM25__{ds}')
    save_result(ds, 'BM25', m)

for ds in DATASETS:
    run_bm25(ds)

In [ ]:
# Cell 6 — Dense retrieval (cần GPU)
import numpy as np
from sentence_transformers import SentenceTransformer

DENSE_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'

def run_dense(ds, model_name=DENSE_MODEL):
    folder = folders[ds]
    corpus = load_corpus(folder)
    qrels = load_qrels(folder)
    queries = filter_queries_by_qrels(load_queries(folder), qrels)
    model = SentenceTransformer(model_name, device='cuda' if torch.cuda.is_available() else 'cpu')
    doc_ids = list(corpus)
    doc_emb = model.encode([f'{t} {x}'.strip() for t, x in (corpus[d] for d in doc_ids)],
                           batch_size=128, show_progress_bar=True, convert_to_numpy=True)
    q_emb = model.encode(list(queries.values()), batch_size=128, convert_to_numpy=True)
    doc_emb /= np.linalg.norm(doc_emb, axis=1, keepdims=True)
    q_emb /= np.linalg.norm(q_emb, axis=1, keepdims=True)
    sim = q_emb @ doc_emb.T
    results = {}
    for i, qid in enumerate(queries):
        results[qid] = [doc_ids[j] for j in np.argsort(-sim[i])[:1000]]
    short = model_name.split('/')[-1]
    m = evaluate(results, qrels, f'{short}__{ds}')
    save_result(ds, short, m)

for ds in DATASETS:
    run_dense(ds)

In [ ]:
# Cell 7 — Cross-encoder re-ranking top-100 của BM25
from sentence_transformers import CrossEncoder

def run_rerank(ds):
    folder = folders[ds]
    corpus = load_corpus(folder)
    qrels = load_qrels(folder)
    queries = filter_queries_by_qrels(load_queries(folder), qrels)
    doc_ids = list(corpus)
    bm25 = BM25Okapi([tokenize(f'{t} {x}') for t, x in (corpus[d] for d in doc_ids)])
    bm25_results = {}
    for qid, qtext in queries.items():
        s = bm25.get_scores(tokenize(qtext))
        bm25_results[qid] = [doc_ids[j] for j in np.argsort(-s)[:100]]
    ce = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', max_length=512)
    results = {}
    for n, qid in enumerate(queries, 1):
        cand = bm25_results[qid]
        pairs = [(queries[qid], f'{corpus[d][0]} {corpus[d][1]}'.strip()) for d in cand]
        ce_scores = ce.predict(pairs, batch_size=64)
        order = sorted(range(len(cand)), key=lambda j: ce_scores[j], reverse=True)
        results[qid] = [cand[j] for j in order]
        if n % 50 == 0:
            print(f'  {n}/{len(queries)}')
    m = evaluate(results, qrels, f'BM25+CE__{ds}')
    save_result(ds, 'BM25+CE', m)

for ds in DATASETS:
    run_rerank(ds)

In [ ]:
# Cell 8 — tổng hợp kết quả thành bảng
import glob

PAPER_NDCG = {
    ('scifact', 'BM25'): 0.665,
    ('nfcorpus', 'BM25'): 0.318,
    ('fiqa', 'BM25'): 0.236,
}

rows = []
for p in sorted(glob.glob(os.path.join(RESULTS_DIR, '*.json'))):
    with open(p) as f:
        rows.append(json.load(f))

print(f"{'Dataset':<12} {'Model':<28} {'nDCG@10':>8} {'Paper':>8} {'Recall@100':>10}")
print('-' * 70)
for r in rows:
    paper = PAPER_NDCG.get((r['dataset'], r['model'].split('+')[0]), '-')
    print(f"{r['dataset']:<12} {r['model']:<28} {r['nDCG@10']:>8.4f} {str(paper):>8} {r.get('Recall@100', '-'):>10}")

import csv
csv_path = os.path.join(RESULTS_DIR, 'summary.csv')
with open(csv_path, 'w', newline='', encoding='utf-8-sig') as f:
    w = csv.writer(f)
    w.writerow(['dataset', 'model', 'nDCG@10', 'Recall@100', 'nDCG@10_paper'])
    for r in rows:
        paper = PAPER_NDCG.get((r['dataset'], r['model'].split('+')[0]), '')
        w.writerow([r['dataset'], r['model'], r['nDCG@10'], r.get('Recall@100', ''), paper])
print('\n[save]', csv_path)

In [ ]:
# Cell 9 — tải kết quả về máy (chỉ chạy được trên Colab)
from google.colab import files
files.download(os.path.join(RESULTS_DIR, 'summary.csv'))